In [ ]:
import os, json, time, math, glob, zipfile, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
device = "cuda"
EOS_ID = 65354

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.g = nn.Parameter(torch.ones(d_model))
    def forward(self, x):
        meanSquare = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(meanSquare + self.eps)
        return x * self.g

def rope_tables(max_seq_len, d_head, base=10000.0):
    assert d_head % 2 == 0
    i = torch.arange(0, d_head // 2).float()
    theta = base ** (-2 * i / d_head)
    m = torch.arange(max_seq_len).float()
    angles = torch.outer(m, theta)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    T = x.size(2)
    cos, sin = cos[:T], sin[:T]
    x1 = x[..., 0::2]; x2 = x[..., 1::2]
    r1 = x1 * cos - x2 * sin
    r2 = x1 * sin + x2 * cos
    return torch.stack([r1, r2], dim=-1).flatten(-2)

In [ ]:
class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.scale = 1 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        cos, sin = rope_tables(max_seq_len, self.d_head, rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        q = apply_rope(q, self.cos, self.sin)
        k = apply_rope(k, self.cos, self.sin)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        scores = (q * self.scale) @ k.transpose(-2, -1)
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        weights = self.attn_dropout(F.softmax(scores, dim=-1))
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))

class MoEFFN(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, top_k, dropout=0.0):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([SwiGLU(d_model, d_ff, dropout) for _ in range(n_experts)])
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        B, T, d = x.shape
        flat = x.reshape(-1, d)
        with torch.autocast("cuda", enabled=False):
            logits = self.router(flat.float())
        probs = F.softmax(logits, dim=-1)
        top_p, top_i = probs.topk(self.top_k, dim=-1)
        top_p = top_p / top_p.sum(-1, keepdim=True)
        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.experts):
            tok, slot = (top_i == e).nonzero(as_tuple=True)
            if tok.numel() == 0: continue
            y = expert(flat[tok])
            out.index_add_(0, tok, (y * top_p[tok, slot, None]).to(out.dtype))
        f = torch.bincount(top_i.flatten(), minlength=self.n_experts).float() / top_i.numel()
        P = probs.mean(0)
        self.aux_loss = self.n_experts * (f * P).sum()
        return out.view(B, T, d)

In [ ]:
@dataclass
class HybridConfig:
    vocab_size: int
    d_model: int
    n_layers: int
    n_heads: int
    n_kv_heads: int
    d_ff: int
    max_seq_len: int = 1024
    dropout: float = 0.0
    rope_base: float = 10000.0
    n_experts: int = 4
    top_k: int = 2
    d_ff_expert: int = 1024

class HybridBlock(nn.Module):
    def __init__(self, cfg, layer_id):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, cfg.rope_base)
        self.norm2 = RMSNorm(cfg.d_model)
        self.is_moe = (layer_id % 2 == 1)
        if self.is_moe:
            self.ffn = MoEFFN(cfg.d_model, cfg.d_ff_expert, cfg.n_experts, cfg.top_k, cfg.dropout)
        else:
            self.ffn = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class HybridLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([HybridBlock(cfg, i) for i in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
    def forward(self, idx):
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x)
        return self.head(self.norm_f(x))

In [ ]:
pkl = glob.glob("/kaggle/input/**/data.pkl", recursive=True)
print("data.pkl found at:", pkl)
src = os.path.dirname([p for p in pkl if "arch4" in p][0])     # the folder holding data.pkl
print("packing:", src, os.listdir(src))                         # expect byteorder, data, data.pkl, version …
dst = "/kaggle/working/arch4_ckpt.pt"
if os.path.exists(dst): os.remove(dst)
with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as z:
    for root, _, files in os.walk(src):
        for f in files:
            full = os.path.join(root, f)
            z.write(full, arcname=os.path.join("arch4_ckpt", os.path.relpath(full, src)))
print(dst, round(os.path.getsize(dst) / 1e9, 2), "GB")

In [ ]:
paths = ["/kaggle/working/arch4_ckpt.pt"]
print("checkpoint found:", paths)
ck = torch.load(paths[0], map_location="cpu")
cfg = HybridConfig(**ck["cfg"])
model = HybridLM(cfg).to(device)
model.load_state_dict(ck["model"])
model.eval()
print("loaded step", ck["step"], "| best val", round(ck["best_val"], 3))
print(f"{sum(p.numel() for p in model.parameters()):,} parameters | expected 191,800,320")
del ck

In [ ]:
%%writefile a1_tokenizer.py
"""Expose your Assignment-1 BPE implementation to Assignment 3.

Reuse your A-1 tokenizer code/class here. The important change is that it must be
RETRAINED on the `train_text` supplied by Assignment 3 rather than reusing a tokenizer
trained on the Assignment-1 dataset.
"""
import re
from collections import Counter

PAT=re.compile(r" ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")
class BPETokeniser:#defining the class
    def __init__(self):
        self.merges={}
        self.vocab={i: bytes([i]) for i in range(256)}
        self.cache={}

    @property
    def vocab_size(self):#vocab size
        return len(self.vocab)

    def _count_pairs(self,words,counts):#how actuall work we start counting how many words have same pair 
        pair_counts={}
        for word,count in zip(words,counts):
            for a,b in zip(word,word[1:]):
                pair_counts[(a,b)]=pair_counts.get((a,b),0)+count
        return pair_counts
    def _merge(self,ids,pair,new_id):
        out=[]
        i=0
        while i<len(ids):
            if i+1<len(ids) and ids[i]==pair[0] and ids[i+1]==pair[1]:
                out.append(new_id)
                i+=2
            else:
                out.append(ids[i])
                i+=1
        return out
    def train(self,text,vocab_size):
        chunks=PAT.findall(text)
        chunk_counts=Counter(chunks)
        words=[]
        counts=[]
        for chunk,n in chunk_counts.items():
            words.append(list(chunk.encode("utf-8")))
            counts.append(n)
        for r in range(vocab_size-256):
                pair_counts=self._count_pairs(words,counts)
                if not pair_counts:
                    break
                best = min(pair_counts, key=lambda p: (-pair_counts[p], p))
                new_id=256+r
                self.merges[best]=new_id
                self.vocab[new_id]=self.vocab[best[0]] + self.vocab[best[1]]
                words=[self._merge(w,best,new_id) for w in words]
    def _encode_chunk(self,chunk):
        if chunk in self.cache:
            return self.cache[chunk]
        ids=list(chunk.encode("utf-8"))
        while True:
            candidates=[p for p in zip(ids,ids[1:]) if p in self.merges]
            if not candidates:
                break
            best=min(candidates,key=lambda p:self.merges[p])
            ids=self._merge(ids,best,self.merges[best])
        self.cache[chunk]=ids
        return ids
    def encode(self,text):
        ids=[]
        for chunk in PAT.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids
    def decode(self,ids):
        data=b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8",errors="replace")
             
         

def train_tokenizer(train_text: str, vocab_size: int):
    """Train your A-1 BPE tokenizer on `train_text` and return the trained tokenizer.

    Required returned interface:
      tokenizer.encode(text) -> sequence of integer ids
      tokenizer.decode(ids) -> string
      tokenizer.vocab_size OR tokenizer.get_vocab_size()

    Replace this stub with your Assignment-1 tokenizer implementation/training call.
    """
    tok=BPETokeniser()
    tok.train(train_text,vocab_size)
    return tok

In [ ]:
import re
from a1_tokenizer import BPETokeniser

In [ ]:
mp = glob.glob("/kaggle/input/**/legal_bpe_merges.json", recursive=True)[0]
raw = json.load(open(mp))
print(type(raw).__name__, len(raw))

In [ ]:
def load_tokenizer(raw):
    tok = BPETokeniser()
    if isinstance(raw, dict) and "merges" in raw:
        raw = raw["merges"]
    triples = []
    if isinstance(raw, dict):                                  # {"a,b": new_id}
        for k, v in raw.items():
            a, b = map(int, re.split(r"[,\s]+", k.strip("()[] ")))
            triples.append((a, b, int(v)))
    else:                                                      # list form
        for i, m in enumerate(raw):
            triples.append((int(m[0]), int(m[1]), int(m[2]) if len(m) == 3 else 256 + i))
    for a, b, new in sorted(triples, key=lambda t: t[2]):      # rebuild in learned order
        tok.merges[(a, b)] = new
        tok.vocab[new] = tok.vocab[a] + tok.vocab[b]
    return tok

tok = load_tokenizer(raw)
print("merges:", len(tok.merges), "| vocab:", tok.vocab_size)   # expect 65,098 | 65,354

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")      # 1. take the card out of the bag
print("token loaded:", HF_TOKEN[:3] + "…", "| length", len(HF_TOKEN))   # expect hf_… and ~37 chars

REPO = "Exploration-Lab/CS-781-Capstone"
path = hf_hub_download(REPO, "Phase_1_Pretraining/Test/test.json",
                       repo_type="dataset", token=HF_TOKEN)                # 2. show it at the gate
test = json.load(open(path))
print(len(test), "test prompts | example id:", list(test)[0])

In [ ]:
TEMPERATURE = 0.7
TOP_K       = 40
REP_PENALTY = 1.2       # your choice (CTRL paper: 1.2; 1.0 = off)
MAX_NEW     = 256
BATCH       = 8

@torch.no_grad()
def generate_batch(prompts):
    x = torch.tensor(prompts, device=device)
    x0 = x.size(1)                                         # prompt length
    for _ in range(MAX_NEW):
        ctx = x[:, -cfg.max_seq_len:]
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(ctx)[:, -1, :].float()
        logits[:, EOS_ID] = -float("inf")
        if REP_PENALTY != 1.0 and x.size(1) > x0:          # penalize tokens already generated
            gen = x[:, x0:]
            sc = logits.gather(1, gen)
            sc = torch.where(sc > 0, sc / REP_PENALTY, sc * REP_PENALTY)
            logits.scatter_(1, gen, sc)
        logits = logits / TEMPERATURE
        if TOP_K:
            v, _ = torch.topk(logits, TOP_K)
            logits[logits < v[:, [-1]]] = -float("inf")
        x = torch.cat([x, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
    return x[:, -MAX_NEW:].tolist()

def to_128_words(ids):
    words = tok.decode(ids).replace("\ufffd", "").split()
    return " ".join(words[:128]), len(words)

ids_list = {k: tok.encode(v["text"]) for k, v in test.items()}
MAXP = cfg.max_seq_len - MAX_NEW                           # 768 prompt tokens max
lens = sorted(min(len(v), MAXP) for v in ids_list.values())
print(f"prompt tokens used: min {lens[0]} | median {lens[len(lens)//2]} | max {lens[-1]} (was 537 for all)")

In [ ]:
torch.manual_seed(0)
keys_t = list(test)[:2]
L = min(min(len(ids_list[k]) for k in keys_t), MAXP)
t = time.time()
outs = generate_batch([ids_list[k][-L:] for k in keys_t])
per = (time.time() - t) / len(keys_t)
for k, o in zip(keys_t, outs):
    text, n = to_128_words(o)
    print(f"[{k}] {n} words → {text[:300]} ...\n")
print(f"{per:.1f} s per prompt (batch of 2, so slower than real) → full set ≈ {per*len(test)/60/3:.0f}–{per*len(test)/60:.0f} min")

In [ ]:
torch.manual_seed(0)
keys = sorted(test, key=lambda k: len(ids_list[k]))        # similar lengths in each batch
pred, short = {}, 0
t0 = time.time()
for i in range(0, len(keys), BATCH):
    chunk = keys[i:i+BATCH]
    L = min(min(len(ids_list[k]) for k in chunk), MAXP)    # as much prompt as this batch allows
    outs = generate_batch([ids_list[k][-L:] for k in chunk])
    for k, o in zip(chunk, outs):
        text, n = to_128_words(o)
        pred[k] = {"text": text}
        short += n < 128
    done = min(i + BATCH, len(keys))
    if done % 80 == 0 or done == len(keys):
        el = time.time() - t0
        print(f"{done}/{len(keys)} done | {el/60:.0f} min | ~{el/done*(len(keys)-done)/60:.0f} min left")

assert set(pred) == set(test), "ID mismatch"
assert all(p["text"].strip() for p in pred.values()), "empty text"
lens = [len(p["text"].split()) for p in pred.values()]
print(f"IDs OK: {len(pred)} | words min {min(lens)}, max {max(lens)} | short (<128): {short}")
with open("/kaggle/working/prediction.json", "w") as f:
    json.dump(pred, f, ensure_ascii=False)
with zipfile.ZipFile("/kaggle/working/submission.zip", "w") as z:
    z.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("submission.zip ready")